# Task B: Alignment using DPO

In [1]:
# Importing the required libraries
import os
import numpy as np
import random
import gc
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, PrefixTuningConfig, get_peft_model
from huggingface_hub import login
from dotenv import load_dotenv
import gc
import torch

# Seed setup
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [2]:
# HF authentication and model/dataset loading
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
hf_username = "sohammandal01"

base_model_id = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading dataset...")
dataset = load_dataset("nrizwan/safe_ai_assignment_1", data_files={"train": "preference_train.csv"})
train_dataset = dataset['train'].shuffle(seed=42).select(range(4000))

def tokenize_dpo(batch):
    chosen_inputs, rejected_inputs = [], []
    chosen_labels, rejected_labels = [], []

    for q, more, less in zip(batch['Question'], batch['More_Prefered'], batch['Less_Prefered']):
        prompt = f"Question: {q}\nAnswer:"
        chosen = f"{prompt} {more}{tokenizer.eos_token}"
        rejected = f"{prompt} {less}{tokenizer.eos_token}"

        tok_chosen = tokenizer(chosen, truncation=True, max_length=256, padding="max_length")
        tok_rejected = tokenizer(rejected, truncation=True, max_length=256, padding="max_length")
        tok_prompt = tokenizer(prompt, truncation=True, max_length=256)

        prompt_len = len(tok_prompt['input_ids'])

        c_label = [
            -100 if (i < prompt_len or token == tokenizer.pad_token_id) else token
            for i, token in enumerate(tok_chosen['input_ids'])
        ]
        r_label = [
            -100 if (i < prompt_len or token == tokenizer.pad_token_id) else token
            for i, token in enumerate(tok_rejected['input_ids'])
        ]

        chosen_inputs.append(tok_chosen['input_ids'])
        chosen_labels.append(c_label)
        rejected_inputs.append(tok_rejected['input_ids'])
        rejected_labels.append(r_label)

    return {
        "chosen_input_ids": chosen_inputs, "chosen_labels": chosen_labels,
        "rejected_input_ids": rejected_inputs, "rejected_labels": rejected_labels
    }

print("Tokenizing DPO dataset...")
dpo_dataset = train_dataset.map(tokenize_dpo, batched=True, batch_size=100)
dpo_dataset.set_format(type="torch",columns=["chosen_input_ids", "chosen_labels", "rejected_input_ids", "rejected_labels"])
print("DPO Dataset ready!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset...
Tokenizing DPO dataset...
DPO Dataset ready!


In [3]:
# DPO data collator
def dpo_data_collator(features):
    return {
        "chosen_input_ids": torch.stack([f["chosen_input_ids"] for f in features]),
        "chosen_labels": torch.stack([f["chosen_labels"] for f in features]),
        "rejected_input_ids": torch.stack([f["rejected_input_ids"] for f in features]),
        "rejected_labels": torch.stack([f["rejected_labels"] for f in features]),
    }

# Custom DPO trainer
class CustomDPOTrainer(Trainer):
    def __init__(self, ref_model=None, beta=0.1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ref_model = ref_model
        self.beta = beta
        if self.ref_model is not None:
            self.ref_model.eval()

    def get_batch_logps(self, logits, labels):
        logits = logits[:, :-1, :].contiguous()
        labels = labels[:, 1:].contiguous()

        loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
        log_probs = -loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
        log_probs = log_probs.view(labels.size())

        loss_mask = labels != -100
        return (log_probs * loss_mask).sum(-1)

    # Computation of loss function
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        policy_chosen_logits = model(input_ids=inputs["chosen_input_ids"]).logits
        policy_rejected_logits = model(input_ids=inputs["rejected_input_ids"]).logits

        policy_chosen_logps = self.get_batch_logps(policy_chosen_logits, inputs["chosen_labels"])
        policy_rejected_logps = self.get_batch_logps(policy_rejected_logits, inputs["rejected_labels"])

        with torch.no_grad():
            if self.ref_model is not None:
                ref_chosen_logits = self.ref_model(input_ids=inputs["chosen_input_ids"]).logits
                ref_rejected_logits = self.ref_model(input_ids=inputs["rejected_input_ids"]).logits
            else:
                with model.disable_adapter():
                    ref_chosen_logits = model(input_ids=inputs["chosen_input_ids"]).logits
                    ref_rejected_logits = model(input_ids=inputs["rejected_input_ids"]).logits

            ref_chosen_logps = self.get_batch_logps(ref_chosen_logits, inputs["chosen_labels"])
            ref_rejected_logps = self.get_batch_logps(ref_rejected_logits, inputs["rejected_labels"])

        pi_logratios = policy_chosen_logps - policy_rejected_logps
        ref_logratios = ref_chosen_logps - ref_rejected_logps

        logits = pi_logratios - ref_logratios
        loss = -F.logsigmoid(self.beta * logits).mean()

        if return_outputs:
            return loss, {"logits": logits}
        return loss

## Full Parameter fine-tuning using DPO

In [ ]:
# Full Parameter fine-tuning using DPO
print("Starting Full Parameter DPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

policy_model_full = AutoModelForCausalLM.from_pretrained(base_model_id, device_map={"": 0})
ref_model_full = AutoModelForCausalLM.from_pretrained(base_model_id, device_map={"": 0})
ref_model_full.eval()

for param in ref_model_full.parameters():
    param.requires_grad = False

policy_model_full.config.use_cache = False

training_args_full = TrainingArguments(
    output_dir="./dpo_model_full",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    remove_unused_columns=False,
    logging_steps=10,
    gradient_checkpointing=True,
    fp16=True,
)

trainer_full = CustomDPOTrainer(
    ref_model=ref_model_full,
    beta=0.1,
    model=policy_model_full,
    args=training_args_full,
    train_dataset=dpo_dataset,
    data_collator=dpo_data_collator,
)

trainer_full.train()
print("Full Parameter DPO Alignment complete!")

repo_full_dpo = f"{hf_username}/gpt2-aligned-dpo-full-assignment"
trainer_full.model.push_to_hub(repo_full_dpo)
tokenizer.push_to_hub(repo_full_dpo)
print(f"Full DPO model pushed to: {repo_full_dpo}")

# Prefix Tuning DPO Alignment

In [ ]:
# Prefix Tuning DPO Alignment
print("Starting Prefix Tuning DPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

base_lm_prefix = AutoModelForCausalLM.from_pretrained(base_model_id, device_map={"": 0})
peft_config_prefix = PrefixTuningConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=20)
policy_model_prefix = get_peft_model(base_lm_prefix, peft_config_prefix)

training_args_prefix = TrainingArguments(
    output_dir="./dpo_model_prefix",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    remove_unused_columns=False,
    logging_steps=10,
    gradient_checkpointing=False,
    fp16=True,
)

trainer_prefix = CustomDPOTrainer(
    ref_model=None,
    beta=0.1,
    model=policy_model_prefix,
    args=training_args_prefix,
    train_dataset=dpo_dataset,
    data_collator=dpo_data_collator,
)

trainer_prefix.train()
print("Prefix Tuning DPO Alignment complete!")

repo_prefix_dpo = f"{hf_username}/gpt2-aligned-dpo-prefix-assignment"
trainer_prefix.model.push_to_hub(repo_prefix_dpo)
tokenizer.push_to_hub(repo_prefix_dpo)
print(f"Prefix DPO model pushed to: {repo_prefix_dpo}")

Starting Prefix Tuning DPO Alignment...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
10,8.254731
20,8.054868
30,9.182523
40,6.239912
50,5.593592
60,5.713542
70,6.176980
80,5.741679
90,2.705631
100,4.305455


Prefix Tuning DPO Alignment complete!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  14%|#4        |  556kB / 3.93MB            

README.md: 0.00B [00:00, ?B/s]

Prefix DPO model pushed to: sohammandal01/gpt2-aligned-dpo-prefix-assignment


# LoRA DPO Alignment

In [ ]:
# LoRA DPO Alignment
print("Starting LoRA DPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

base_lm_lora = AutoModelForCausalLM.from_pretrained(base_model_id, device_map={"": 0})
peft_config_lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32, lora_dropout=0.1)
policy_model_lora = get_peft_model(base_lm_lora, peft_config_lora)

policy_model_lora.config.use_cache = False

training_args_lora = TrainingArguments(
    output_dir="./dpo_model_lora",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    remove_unused_columns=False,
    logging_steps=10,
    gradient_checkpointing=False,
    fp16=False,
)

trainer_lora = CustomDPOTrainer(
    ref_model=None,
    beta=0.1,
    model=policy_model_lora,
    args=training_args_lora,
    train_dataset=dpo_dataset,
    data_collator=dpo_data_collator,
)

trainer_lora.train()
print("LoRA Finetuning Complete!")

repo_lora_dpo = f"{hf_username}/gpt2-aligned-dpo-lora-assignment"
trainer_lora.model.push_to_hub(repo_lora_dpo)
tokenizer.push_to_hub(repo_lora_dpo)
print(f"LoRA DPO model pushed to: {repo_lora_dpo}")

Starting LoRA DPO Alignment...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Step,Training Loss
10,1.421958
20,1.745204
30,1.579969
40,1.210732
50,1.434427
60,1.410134
70,1.304432
80,1.404554
90,1.333188
100,1.346641


LoRA Finetuning Complete!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  18%|#7        |  563kB / 3.15MB            

README.md: 0.00B [00:00, ?B/s]

LoRA DPO model pushed to: sohammandal01/gpt2-aligned-dpo-lora-assignment


# QLoRA DPO Alignment

In [4]:
# QLoRA DPO Alignment
print("Starting QLoRA DPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_lm_qlora = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map={"": 0})
peft_config_qlora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32, lora_dropout=0.1)
policy_model_qlora = get_peft_model(base_lm_qlora, peft_config_qlora)

training_args_qlora = TrainingArguments(
    output_dir="./dpo_model_qlora",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    remove_unused_columns=False,
    logging_steps=10,
    gradient_checkpointing=True,
    fp16=True,
)

trainer_qlora = CustomDPOTrainer(
    ref_model=None,
    beta=0.1,
    model=policy_model_qlora,
    args=training_args_qlora,
    train_dataset=dpo_dataset,
    data_collator=dpo_data_collator,
)

trainer_qlora.train()
print("QLoRA Finetuning Complete!")

Starting QLoRA DPO Alignment...


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step,Training Loss
10,1.599970
20,1.933542
30,1.559094
40,1.412111
50,1.631796
60,1.593223
70,1.483966
80,1.281683
90,1.309451
100,1.359233


Step,Training Loss
10,1.599970
20,1.933542
30,1.559094
40,1.412111
50,1.631796
60,1.593223
70,1.483966
80,1.281683
90,1.309451
100,1.359233


QLoRA Finetuning Complete!


In [5]:
repo_qlora_dpo = f"{hf_username}/gpt2-aligned-dpo-qlora-assignment"
trainer_qlora.model.push_to_hub(repo_qlora_dpo)
tokenizer.push_to_hub(repo_qlora_dpo)
print(f"QLoRA DPO model pushed to: {repo_qlora_dpo}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  17%|#7        |  537kB / 3.15MB            

README.md: 0.00B [00:00, ?B/s]

QLoRA DPO model pushed to: sohammandal01/gpt2-aligned-dpo-qlora-assignment


In [6]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
